# The installing packages

In [2]:
!pip install lightning
!pip install torchmetrics
!pip install kagglehub

# Download Dataset

In [7]:
import kagglehub

path = kagglehub.dataset_download("sukhmandeepsinghbrar/housing-price-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'housing-price-dataset' dataset.
Path to dataset files: /kaggle/input/housing-price-dataset


# Import Library

In [34]:
import torch
import torch.nn as nn
import lightning as L
from torchmetrics import Accuracy
import torch.nn.functional as F
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from lightning.pytorch.callbacks import ModelCheckpoint , EarlyStopping

# Dataset

In [27]:
df = pd.read_csv(
    "/root/.cache/kagglehub/datasets/sukhmandeepsinghbrar/housing-price-dataset/versions/Housing.csv"
)

print(df.head())
print(df.dtypes)
print(df.select_dtypes(include="object").columns.tolist())

           id             date     price  ...     long  sqft_living15  sqft_lot15
0  7229300521  20141013T000000  231300.0  ... -122.257           1340        5650
1  6414100192  20141209T000000  538000.0  ... -122.319           1690        7639
2  5631500400  20150225T000000  180000.0  ... -122.233           2720        8062
3  2487200875  20141209T000000  604000.0  ... -122.393           1360        5000
4  1954400510  20150218T000000  510000.0  ... -122.045           1800        7503

[5 rows x 21 columns]
id                 int64
date              object
price            float64
bedrooms           int64
bathrooms        float64
sqft_living        int64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
condition          int64
grade              int64
sqft_above         int64
sqft_basement      int64
yr_built           int64
yr_renovated       int64
zipcode            int64
lat              float64
long             float64
sqft_livin

In [30]:
class HousingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]


class HousingDataModule(L.LightningDataModule):
    def __init__(
        self,
        csv_path,
        batch_size=32,
        num_workers=0
    ):
        super().__init__()

        self.csv_path = csv_path
        self.batch_size = batch_size
        self.num_workers = num_workers

        self.scaler = StandardScaler()

    def setup(self, stage=None):
        if stage == "predict":
            return

        # Load data
        df = pd.read_csv(self.csv_path)

        # Extract date features
        date = pd.to_datetime(
            df["date"],
            format="%Y%m%dT%H%M%S"
        )

        df["sale_year"] = date.dt.year
        df["sale_month"] = date.dt.month
        df["sale_day"] = date.dt.day

        # Remove ID and original date
        df = df.drop(columns=["id", "date"])

        # Separate features and target
        X = df.drop(columns=["price"])
        y = df["price"]

        # Train: 80%, temporary: 20%
        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        # Validation: 10%, test: 10%
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=0.5, random_state=42
        )

        # Fit scaler only on training features
        X_train = self.scaler.fit_transform(X_train)
        X_val = self.scaler.transform(X_val)
        X_test = self.scaler.transform(X_test)

        # Create datasets
        self.train_dataset = HousingDataset(
            X_train, y_train.to_numpy().reshape(-1, 1)
        )

        self.val_dataset = HousingDataset(
            X_val, y_val.to_numpy().reshape(-1, 1)
        )

        self.test_dataset = HousingDataset(
            X_test, y_test.to_numpy().reshape(-1, 1)
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers
        )


# Model

In [31]:
class HousingModel(L.LightningModule):
    def __init__(
        self,
        input_dim=21,
        learning_rate=1e-3
    ):
        super().__init__()

        self.save_hyperparameters()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

        self.criterion = nn.MSELoss()

    def forward(self, x):
        return self.network(x)

    def training_step(self, batch, batch_idx):
        X, y = batch

        predictions = self(X)
        loss = self.criterion(predictions, y)

        self.log(
            "train_loss",
            loss,
            on_step=False,
            on_epoch=True,
            prog_bar=True
        )

        return loss

    def validation_step(self, batch, batch_idx):
        X, y = batch

        predictions = self(X)
        loss = self.criterion(predictions, y)

        mae = torch.mean(torch.abs(predictions - y))

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_mae", mae, prog_bar=True)

    def test_step(self, batch, batch_idx):
        X, y = batch

        predictions = self(X)
        loss = self.criterion(predictions, y)
        mae = torch.mean(torch.abs(predictions - y))

        self.log("test_loss", loss)
        self.log("test_mae", mae)

    def configure_optimizers(self):
        return torch.optim.Adam(
            self.parameters(),
            lr=self.hparams.learning_rate
        )


# Configure



In [32]:
CSV_PATH = "/root/.cache/kagglehub/datasets/sukhmandeepsinghbrar/housing-price-dataset/versions/Housing.csv"

BATCH_SIZE = 32
NUM_WORKERS = 0

LEARNING_RATE = 1e-3

INPUT_DIM = 21

# Train

In [35]:
torch.manual_seed(123)

data_module = HousingDataModule(
    csv_path=CSV_PATH,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS
)

model = HousingModel(input_dim=INPUT_DIM, learning_rate=LEARNING_RATE)

callbacks = [
    ModelCheckpoint(
        monitor="val_loss",
        mode="min",
        save_top_k=10,
        save_last=True
    ),
    EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=10,
        min_delta=0.0,
        verbose=True
    )
]

trainer = L.Trainer(
    max_epochs=100,
    accelerator="auto",
    devices="auto",
    callbacks=callbacks
)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [37]:
trainer.fit(model , data_module)

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ network   │ Sequential │ 13.2 K │ train │     0 │
│ 1 │ criterion │ MSELoss    │      0 │ train │     0 │
└───┴───────────┴────────────┴────────┴───────┴───────┘

Trainable params: 13.2 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 13.2 K                                                                                               
Total estimated model params size (MB): 0.053                                                                      
Modules in train mode: 9                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO: Metric val_loss improved. New best score: 163105046528.000
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_loss improved. New best score: 163105046528.000
INFO: Metric val_loss improved by 93932396544.000 >= min_delta = 0.0. New best score: 69172649984.000
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_loss improved by 93932396544.000 >= min_delta = 0.0. New best score: 69172649984.000
INFO: Metric val_loss improved by 12819791872.000 >= min_delta = 0.0. New best score: 56352858112.000
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_loss improved by 12819791872.000 >= min_delta = 0.0. New best score: 56352858112.000
INFO: Metric val_loss improved by 8624390144.000 >= min_delta = 0.0. New best score: 47728467968.000
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_loss improved by 8624390144.000 >= min_delta = 0.0. New best score: 47728467968.000
INFO: Metric val_loss improved by 6187036672.000 >= min_delta = 0.0. New best score: 4154143

In [44]:
trainer.test(model , data_module.test_dataloader())

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │       30188969984.0       │
│         test_mae          │      100272.7734375       │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 30188969984.0, 'test_mae': 100272.7734375}]

In [45]:
trainer.validate(model , data_module.val_dataloader())

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         val_loss          │       25868824576.0       │
│          val_mae          │       95729.703125        │
└───────────────────────────┴───────────────────────────┘

[{'val_loss': 25868824576.0, 'val_mae': 95729.703125}]

In [46]:
trainer.save_checkpoint("model.ckpt")

INFO: `weights_only` was not set, defaulting to `False`.
INFO:lightning.pytorch.trainer.connectors.checkpoint_connector:`weights_only` was not set, defaulting to `False`.
